# Face Recognition

## Import Libraries

In [1]:
# Import standard dependencies
import cv2 #for accessing webcam and for image processing
import os #Directory management 
import random 
import numpy as np
from matplotlib import pyplot as plt  #reconstruct images


In [ ]:
# Import tensorflow dependencies - Functional API
from tensorflow.keras.models import Model 
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten
import tensorflow as tf

### Set GPU Growth

In [3]:
# Avoid OOM (Out of memory) errors by setting GPU Memory Consumption Growth
gpus = tf.config.experimental.list_physical_devices('GPU') #scanning for GPU
for gpu in gpus: 
    tf.config.experimental.set_memory_growth(gpu, True) #setting Growth limit

In [4]:
# Setup 3 paths for the 3 types of images
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

In [5]:
# # Make the directories/folders
# os.makedirs(POS_PATH)
# os.makedirs(NEG_PATH)
# os.makedirs(ANC_PATH)

# Adding training dataset

In [6]:
# Uncompress the "Labelled Faces in the Wild" Dataset (the negative dataset)
# !tar -xf lfw.tgz

In [7]:
# # Move LFW Images to the following repository data/negative
# for directory in os.listdir('lfw'):
#     for file in os.listdir(os.path.join('lfw', directory)):
#         EX_PATH = os.path.join('lfw', directory, file)
#         NEW_PATH = os.path.join(NEG_PATH, file)
#         os.replace(EX_PATH, NEW_PATH)

# Implementing the feature of collecting images

In [8]:
# Import uuid library to generate unique image names for achor and positive
import uuid
os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))

'data\\anchor\\cdb0481f-0121-11ef-a7ba-6c2408fbed74.jpg'

In [9]:
# Establish a connection to the webcam
cap = cv2.VideoCapture(0)
while cap.isOpened(): 
    ret, frame = cap.read()

    # Cut down frame to 250x250px
    frame = frame[120:120+250,200:200+250, :]
    
    # Collect anchors 
    if cv2.waitKey(1) & 0XFF == ord('a'):
        # Create the unique file path 
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        # Write out anchor image
        cv2.imwrite(imgname, frame)
    
    # Collect positives
    if cv2.waitKey(1) & 0XFF == ord('p'):
        # Create the unique file path 
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        # Write out positive image
        cv2.imwrite(imgname, frame)

    
    # Show image back to screen
    cv2.imshow('Image Collection', frame)
    
    # Breaking gracefully
    if cv2.waitKey(1) & 0XFF == ord('q'):
        break
        
# Release the webcam
cap.release()
# Close the image show frame
cv2.destroyAllWindows()

In [ ]:
plt.imshow(frame)

## Load and Preprocess Images

In [11]:
# Get Image Directories
anchor = tf.data.Dataset.list_files(ANC_PATH+'\*.jpg').take(300)
positive = tf.data.Dataset.list_files(POS_PATH+'\*.jpg').take(300)
negative = tf.data.Dataset.list_files(NEG_PATH+'\*.jpg').take(300)

In [12]:
dir_test = anchor.as_numpy_iterator()

In [13]:
print(dir_test.next())

b'data\\anchor\\ae20ad36-0120-11ef-bc5f-6c2408fbed74.jpg'


## Preprocessing

In [14]:
def preprocess(file_path):
    
    # Read in image from file path
    byte_img = tf.io.read_file(file_path)
    # Load in the image 
    img = tf.io.decode_jpeg(byte_img)
    
    # Preprocessing steps - resizing the image to be 100x100x3
    img = tf.image.resize(img, (100,100))
    # Scale image to be between 0 and 1 
    img = img / 255.0

    # Return image
    return img

In [15]:
img = preprocess('data\\anchor\\85fd2dad-0120-11ef-bc69-6c2408fbed74.jpg')
img

<tf.Tensor: shape=(100, 100, 3), dtype=float32, numpy=
array([[[0.3529412 , 0.3529412 , 0.34509805],
        [0.3529412 , 0.3529412 , 0.34509805],
        [0.3529412 , 0.3529412 , 0.3509804 ],
        ...,
        [0.36372548, 0.36372548, 0.36372548],
        [0.3629902 , 0.3629902 , 0.3629902 ],
        [0.3647059 , 0.3647059 , 0.37254903]],

       [[0.35686275, 0.35686275, 0.35882354],
        [0.35686275, 0.35686275, 0.35882354],
        [0.35686275, 0.35686275, 0.35882354],
        ...,
        [0.3647059 , 0.3647059 , 0.35882354],
        [0.3647059 , 0.3647059 , 0.3647059 ],
        [0.3647059 , 0.3647059 , 0.3647059 ]],

       [[0.36078432, 0.35784313, 0.3745098 ],
        [0.36078432, 0.35784313, 0.3745098 ],
        [0.36078432, 0.35784313, 0.3745098 ],
        ...,
        [0.36691177, 0.36691177, 0.36544117],
        [0.36691177, 0.36691177, 0.36691177],
        [0.3647059 , 0.3647059 , 0.3647059 ]],

       ...,

       [[0.71960783, 0.7352941 , 0.7794118 ],
        [0.71

In [16]:
img.numpy().max()

0.95392156

In [ ]:
plt.imshow(img)

## Creating a Labelled Dataset

In [18]:
# (anchor, positive) => 1,1,1,1,1
# (anchor, negative) => 0,0,0,0,0

In [19]:
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [20]:
samples = data.as_numpy_iterator()

In [21]:
example = samples.next()

In [22]:
example

(b'data\\anchor\\ae86347f-0120-11ef-8939-6c2408fbed74.jpg',
 b'data\\positive\\15440321-0121-11ef-8c0c-6c2408fbed74.jpg',
 1.0)

In [23]:
def preprocess_twin(input_img, validation_img, label):
    return(preprocess(input_img), preprocess(validation_img), label)

In [24]:
res = preprocess_twin(*example)

In [ ]:
plt.imshow(res[1])

In [26]:
res[2]

1.0

In [27]:
# Build dataloader pipeline
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size=10000)

In [28]:
# Training partition
train_data = data.take(round(len(data)*.7))
train_data = train_data.batch(16)
train_data = train_data.prefetch(8)

In [29]:
# Testing partition
test_data = data.skip(round(len(data)*.7))
test_data = test_data.take(round(len(data)*.3))
test_data = test_data.batch(16)
test_data = test_data.prefetch(8)

## Model Engineering

In [30]:
# # comparing with the results from the research paper
# inp = Input(shape=(105,105,3), name='input_image')

# c1 = Conv2D(64, (10,10), activation='relu')(inp)
# m1 = MaxPooling2D(64, (2,2), padding='same')(c1)

# c2 = Conv2D(128, (7,7), activation='relu')(m1)
# m2 = MaxPooling2D(64, (2,2), padding='same')(c2)

# c3 = Conv2D(128, (4,4), activation='relu')(m2)
# m3 = MaxPooling2D(64, (2,2), padding='same')(c3)

# c4 = Conv2D(256, (4,4), activation='relu')(m3)
# f1 = Flatten()(c4)
# d1 = Dense(4096, activation='sigmoid')(f1)

# mod = Model(inputs=[inp], outputs=[d1], name='embedding')
# mod.summary()

In [31]:
def make_embedding(): 
    inp = Input(shape=(100,100,3), name='input_image')
    
    # First block
    c1 = Conv2D(64, (10,10), activation='relu')(inp)
    m1 = MaxPooling2D(64, (2,2), padding='same')(c1)
    
    # Second block
    c2 = Conv2D(128, (7,7), activation='relu')(m1)
    m2 = MaxPooling2D(64, (2,2), padding='same')(c2)
    
    # Third block 
    c3 = Conv2D(128, (4,4), activation='relu')(m2)
    m3 = MaxPooling2D(64, (2,2), padding='same')(c3)
    
    # Final embedding block
    c4 = Conv2D(256, (4,4), activation='relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation='sigmoid')(f1)
    
    
    return Model(inputs=[inp], outputs=[d1], name='embedding')

In [ ]:
embedding = make_embedding()

In [33]:
embedding.summary()

Model: "embedding"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 100, 100, 3)]     0         
                                                                 
 conv2d (Conv2D)             (None, 91, 91, 64)        19264     
                                                                 
 max_pooling2d (MaxPooling2  (None, 46, 46, 64)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 40, 40, 128)       401536    
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 20, 20, 128)       0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 17, 17, 128)       26

## Distance Layer

In [34]:
# Siamese L1 Distance class
class L1Dist(Layer):
    
    # Init method - inheritance
    def __init__(self, **kwargs):
        super().__init__()
       
    # Magic happens here - similarity calculation
    def call(self, input_embedding, validation_embedding):
        return tf.math.abs(input_embedding - validation_embedding)

## Make Siamese Model

In [35]:
def make_siamese_model(): 
    
    # Anchor image input in the network
    input_image = Input(name='input_img', shape=(100,100,3))
    
    # Validation image in the network 
    validation_image = Input(name='validation_img', shape=(100,100,3))
    
    # Combine siamese distance components
    siamese_layer = L1Dist()
    siamese_layer._name = 'distance'
    distances = siamese_layer(embedding(input_image), embedding(validation_image))
    
    # Classification layer 
    classifier = Dense(1, activation='sigmoid')(distances)
    
    return Model(inputs=[input_image, validation_image], outputs=classifier, name='SiameseNetwork')

In [36]:
siamese_model = make_siamese_model()

In [37]:
siamese_model.summary()

Model: "SiameseNetwork"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_img (InputLayer)      [(None, 100, 100, 3)]        0         []                            
                                                                                                  
 validation_img (InputLayer  [(None, 100, 100, 3)]        0         []                            
 )                                                                                                
                                                                                                  
 embedding (Functional)      (None, 4096)                 3896044   ['input_img[0][0]',           
                                                          8          'validation_img[0][0]']      
                                                                                     

## Set up Loss Function and Train step Function

In [38]:
binary_cross_loss = tf.losses.BinaryCrossentropy()

In [39]:
#optimizer
opt = tf.keras.optimizers.Adam(1e-4) # 0.0001

In [40]:
# Establishing Checkpoints
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(opt=opt, siamese_model=siamese_model)

In [41]:
#train step function
test_batch = train_data.as_numpy_iterator()

In [42]:
batch_1 = test_batch.next()

In [43]:
@tf.function
def train_step(batch):
    
    # Record all of our operations 
    with tf.GradientTape() as tape:     
        # Get anchor and positive/negative image
        X = batch[:2]
        # Get label
        y = batch[2]
        
        # Forward pass
        yhat = siamese_model(X, training=True)
        # Calculate loss
        loss = binary_cross_loss(y, yhat)
    print(loss)
        
    # Calculate gradients
    grad = tape.gradient(loss, siamese_model.trainable_variables)
    
    # Calculate updated weights and apply to siamese model
    opt.apply_gradients(zip(grad, siamese_model.trainable_variables))
        
    # Return loss
    return loss

In [44]:
#Loop Training
# Import metric calculations
from tensorflow.keras.metrics import Precision, Recall

In [45]:
def train(data, EPOCHS):
    # Loop through epochs
    for epoch in range(1, EPOCHS+1):
        print('\n Epoch {}/{}'.format(epoch, EPOCHS))
        progbar = tf.keras.utils.Progbar(len(data))
        
        # Creating a metric object 
        r = Recall()
        p = Precision()
        
        # Loop through each batch
        for idx, batch in enumerate(data):
            # Run train step here
            loss = train_step(batch)
            yhat = siamese_model.predict(batch[:2])
            r.update_state(batch[2], yhat)
            p.update_state(batch[2], yhat) 
            progbar.update(idx+1)
        print(loss.numpy(), r.result().numpy(), p.result().numpy())
        
        # Save checkpoints
        if epoch % 10 == 0: 
            checkpoint.save(file_prefix=checkpoint_prefix)

## Train Model

In [46]:
EPOCHS = 50

In [47]:
train(train_data, EPOCHS)


 Epoch 1/50
Tensor("binary_crossentropy/weighted_loss/value:0", shape=(), dtype=float32)
Tensor("binary_crossentropy/weighted_loss/value:0", shape=(), dtype=float32)
27/27 [==============================] - 61s 2s/step
0.1970914 0.5116279 0.95652175

 Epoch 2/50
27/27 [==============================] - 57s 2s/step
0.18541759 0.884058 1.0

 Epoch 3/50
27/27 [==============================] - 59s 2s/step
0.19450973 0.9768519 0.9859813

 Epoch 4/50
27/27 [==============================] - 58s 2s/step
0.0006962643 0.98039216 0.99502486

 Epoch 5/50
27/27 [==============================] - 58s 2s/step
0.23359355 0.93564355 0.97927463

 Epoch 6/50
27/27 [==============================] - 59s 2s/step
0.36082548 0.94711536 1.0

 Epoch 7/50
27/27 [==============================] - 58s 2s/step
0.0011127386 0.9150943 0.98477155

 Epoch 8/50
27/27 [==============================] - 59s 2s/step
0.004027811 0.99065423 1.0

 Epoch 9/50
27/27 [==============================] - 58s 2s/step
0.05429021 

## Evaluate the Model

In [48]:
# Import metric calculations
from tensorflow.keras.metrics import Precision, Recall

In [49]:
# Get a batch of test data, format: input images__validation/positive/negative images___Labels
test_input, test_val, y_true = test_data.as_numpy_iterator().next() 

In [50]:
y_hat = siamese_model.predict([test_input, test_val])

1/1 [==============================] - 0s 423ms/step


In [51]:
# Post processing the results 
resul = []
for prediction in y_hat:
    if prediction >0.5:
        resul.append(1)
    else:
        resul.append(0)
resul

[0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1]

In [52]:
y_true #similar to label

array([0., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1.],
      dtype=float32)

In [53]:
#comparing the above labels automatically

#creating a metric object
m = Recall()

#calculating the recall value
m.update_state(y_true,y_hat)

#printing reults via .results()
m.result().numpy()

1.0

In [54]:
#comparing the above labels automatically

#creating a metric object
m = Precision()

#calculating the Precision value
m.update_state(y_true,y_hat)

#printing reults via .results()
m.result().numpy()

1.0

## Vizualizing Results

In [ ]:
# Set plot size 
plt.figure(figsize=(7,7))

# Set first subplot
plt.subplot(1,2,1) #column 1
plt.imshow(test_input[2]) #input image

# Set second subplot
plt.subplot(1,2,2) #column 2
plt.imshow(test_val[2]) #validation image(positive/Negative)

# Renders cleanly
plt.show()

## Save Model

In [ ]:
# Save weights
siamese_model.save('siamesemodelv2.h5')

In [57]:
# Reload model format: model name, custom layers
siamese_model = tf.keras.models.load_model('siamesemodelv2.h5', 
                                   custom_objects={'L1Dist':L1Dist, 'BinaryCrossentropy':tf.losses.BinaryCrossentropy})

In [58]:
# Make predictions with reloaded model
siamese_model.predict([test_input, test_val])

1/1 [==============================] - 0s 487ms/step


array([[4.34213604e-07],
       [9.99964476e-01],
       [2.74198992e-08],
       [9.99792099e-01],
       [1.00570404e-07],
       [1.00000000e+00],
       [9.96976912e-01],
       [9.99999583e-01],
       [1.00000000e+00],
       [1.00000000e+00],
       [1.00000000e+00],
       [2.33251685e-09],
       [9.98443067e-01],
       [1.00000000e+00],
       [1.00000000e+00],
       [9.99757409e-01]], dtype=float32)

In [59]:
siamese_model.summary()

Model: "SiameseNetwork"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_img (InputLayer)      [(None, 100, 100, 3)]        0         []                            
                                                                                                  
 validation_img (InputLayer  [(None, 100, 100, 3)]        0         []                            
 )                                                                                                
                                                                                                  
 embedding (Functional)      (None, 4096)                 3896044   ['input_img[0][0]',           
                                                          8          'validation_img[0][0]']      
                                                                                     

## Real Time Test

### Verification function

In [60]:
def verify(model, detection_threshold, verification_threshold):
    # Build results array
    results = []
    for image in os.listdir(os.path.join('application_data', 'verification_images')):
        input_img = preprocess(os.path.join('application_data', 'input_image', 'input_image.jpg'))
        validation_img = preprocess(os.path.join('application_data', 'verification_images', image))
        
        # Make Predictions 
        result = model.predict(list(np.expand_dims([input_img, validation_img], axis=1)))
        results.append(result)
    
    # Detection Threshold: Metric above which a prediciton is considered positive 
    detection = np.sum(np.array(results) > detection_threshold)
    
    # Verification Threshold: Proportion of positive predictions / total positive samples 
    verification = detection / len(os.listdir(os.path.join('application_data', 'verification_images'))) 
    verified = verification > verification_threshold
    
    return results, verified

### OpenCV Real Time Verification

In [61]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    frame = frame[120:120+250,200:200+250, :]
    
    cv2.imshow('Verification', frame)
    
    # Verification trigger
    if cv2.waitKey(10) & 0xFF == ord('v'):
        # Save input image to application_data/input_image folder 
#         hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
#         h, s, v = cv2.split(hsv)

#         lim = 255 - 10
#         v[v > lim] = 255
#         v[v <= lim] -= 10
        
#         final_hsv = cv2.merge((h, s, v))
#         img = cv2.cvtColor(final_hsv, cv2.COLOR_HSV2BGR)

        cv2.imwrite(os.path.join('application_data', 'input_image', 'input_image.jpg'), frame)
        # Run verification
        results, verified = verify(siamese_model, 0.5, 0.5)
        print(verified)
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break 
cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 49ms/step
False
1/1 [==============================] - 0s 51ms/step
False
1/1 [==============================] - 0s 49ms/step
False
1/1 [==============================] - 0s 51ms/step
True
